# 02 - Encoding: from RNA to qubits

**Goal:** turn the folding problem into a QUBO, one qubit per decision.

**The naive idea:** one qubit per possible base pair. This blows up. For a
30-base sequence you already need around 130 qubits, which no simulator can
handle.

**The better idea (what we use):** one qubit per *stem*. A stem is a run of
consecutive pairs, like a zipper. Real structures are made of a handful of
stems, so this cuts the qubit count by a factor of 5 to 10.

Qubit k = 1 means "use stem k". The energy is then

    E(x) = sum_k h[k] x[k]  +  sum_{k<m} J[k,m] x[k] x[m]

`h[k]` is the free energy of stem k alone (negative is good).
`J[k,m]` is a big penalty if stems k and m cannot coexist.

### How to run this notebook

This notebook runs on its own. You do not need the others open.

1. Put **gqe-rna.zip** at the top level of your Google Drive (My Drive).
   Do this once. You never need to unzip it yourself.
2. In Colab: **File -> Upload notebook** and pick this file.
3. **Runtime -> Change runtime type -> T4 GPU** (free tier is fine).
4. **Runtime -> Run all**. Approve the Drive popup when it appears.

Results are saved into `gqe-rna/results/` **in your Drive**, so they survive
when the runtime shuts down and the next notebook picks them up automatically.

If a cell says a file is missing, run the notebook it names first.

In [1]:
# STEP 1 - Connect to Drive and find the project.
# Put gqe-rna.zip at the top level of My Drive. Do not unzip it yourself.
import sys, os, glob, subprocess, importlib

from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive"
REPO = os.path.join(ROOT, "gqe-rna")

# Unzip once, if the folder is not there yet.
if not os.path.isdir(os.path.join(REPO, "src")):
    zp = os.path.join(ROOT, "gqe-rna.zip")
    assert os.path.exists(zp), (
        "gqe-rna.zip was not found at the top level of My Drive. "
        "Upload it there (not in a subfolder), then run this cell again.")
    subprocess.run(["unzip", "-q", "-o", zp, "-d", ROOT], check=True)

# If the zip nested an extra folder, find the real one.
if not os.path.isdir(os.path.join(REPO, "src")):
    hits = glob.glob(os.path.join(ROOT, "**", "src", "colab_utils.py"),
                     recursive=True)
    assert hits, "Could not find src/ anywhere in My Drive after unzipping."
    REPO = os.path.dirname(os.path.dirname(hits[0]))

sys.path.insert(0, REPO)
importlib.invalidate_caches()   # Python cached this folder before it existed
from src import colab_utils as cu

cu.setup(repo=REPO)   # chdir into the project so results/ is saved to Drive
cu.install_vienna()

Mounted at /content/drive
project folder: /content/drive/MyDrive/gqe-rna
installing ViennaRNA ...


True

In [2]:
# STEP 2 - Load the project code.
import numpy as np
import matplotlib.pyplot as plt
from src import rna, energy, simulator, pools, metrics, baselines

rna.set_vienna_defaults()      # pin temperature and energy model
print("ViennaRNA available:", rna.HAVE_VIENNA)
cu.show_results()              # what earlier notebooks already saved

ViennaRNA available: True
file                        size   from notebook
--------------------------------------------------
refs.pkl                    0.5K   01


In [3]:
saved = cu.require("refs", made_by="01")
SEQUENCES, refs = saved["sequences"], saved["refs"]
seq = SEQUENCES["challenge44"]
print(seq)

GGAGCAAAACUUGUCGAUUGAGAACAAAAUACAGAAUUUGCUUG


## Finding stems

`min_len` is the shortest stem we allow. It is the main knob controlling how
many qubits we need. Bigger `min_len` = fewer qubits = faster but less accurate.

In [4]:
for m in (2, 3, 4, 5):
    st = rna.find_stems(seq, min_len=m)
    print(f"min_len={m}:  {len(st):3d} stems  ->  {len(st)} qubits")

min_len=2:   80 stems  ->  80 qubits
min_len=3:   27 stems  ->  27 qubits
min_len=4:   10 stems  ->  10 qubits
min_len=5:    1 stems  ->  1 qubits


`min_len=4` gives 10 qubits: tiny and fast. `min_len=3` gives 27, which is at
the edge of what we can batch-simulate. We use 4 for the main run and discuss
3 in notebook 05.

In [5]:
# min_len=4 misses any 3 bp helix, and real MFEs use those.
# So we go down to 3 and trim to a qubit count the simulator can handle.
MIN_LEN, MAX_N = 3, 22        # 22 qubits ~ 540 MB per batch, fine on a T4

stems, h, J = rna.build_encoding(seq, min_len=MIN_LEN, max_n=MAX_N)
n = len(stems)
for k, s in enumerate(stems):
    print(f"  qubit {k:2d}: pairs {s['i']:2d}-{s['j']:2d}, length {s['length']}")

min_len=3: 27 stems found
trimmed to the 22 with the strongest stacking
-> 22 qubits
  qubit  0: pairs  1-42, length 7
  qubit  1: pairs  3-13, length 4
  qubit  2: pairs  9-21, length 4
  qubit  3: pairs 10-26, length 4
  qubit  4: pairs 10-33, length 4
  qubit  5: pairs 15-37, length 4
  qubit  6: pairs 24-39, length 4
  qubit  7: pairs 26-38, length 4
  qubit  8: pairs  1-11, length 3
  qubit  9: pairs  1-38, length 3
  qubit 10: pairs  4-19, length 3
  qubit 11: pairs  4-43, length 3
  qubit 12: pairs  6-38, length 3
  qubit 13: pairs 11-30, length 3
  qubit 14: pairs 14-39, length 3
  qubit 15: pairs 14-43, length 3
  qubit 16: pairs 16-29, length 3
  qubit 17: pairs 20-38, length 3
  qubit 18: pairs 21-38, length 3
  qubit 19: pairs 24-43, length 3
  qubit 20: pairs 31-39, length 3
  qubit 21: pairs 31-43, length 3


### Coverage check: run this before blaming the optimiser

The most common encoding mistake is that the helix you need was filtered out
by `min_len`, so no optimiser could ever find it. This cell says whether every
helix of the true MFE is representable at all.

With `min_len=4` the 3 bp inner helix of our test sequence is missing, and the
gap can never go below about 2 kcal/mol no matter how good the search is.

In [6]:
missing = rna.check_mfe_coverage(seq, stems, ref_db=refs['challenge44'][0])

# If anything is missing, follow the advice printed above and re-run
# the cell before this one. Do not move on to notebook 03 with a gap
# you have not explained.

  helix   1- 42 length 7   in our stem set
  helix  10- 33 length 4   in our stem set
  helix  17- 26 length 3   MISSING

  1 helix/helices missing. Shortest is 3 bp.
  -> these exist at this min_len but were trimmed out.
     Raise max_n in build_encoding.


### Tuning the qubit count

`min_len` moves in big jumps (10 qubits, then 27). If you want a specific
number, `limit_stems` keeps only the N most stabilising stems.

In [7]:
# Example: force exactly 16 qubits using min_len=3.
stems3 = rna.find_stems(seq, min_len=3)
h3, _ = rna.build_qubo(seq, stems3)
small, kept = rna.limit_stems(stems3, h3, max_n=16)
print(f'{len(stems3)} stems -> kept {len(small)}')
print('Keep MIN_LEN=4 below for the fast main run.')

27 stems -> kept 16
Keep MIN_LEN=4 below for the fast main run.


## Building the QUBO

Two stems clash if they share a base (one base can only pair once) or if
they cross each other (a *pseudoknot*, which the standard model excludes).
Clashing pairs get a big positive penalty.

**For stems that CAN coexist we must not put zero.** Stem energies do not
add up. ViennaRNA charges a lone stem about +4 kcal/mol for closing a
hairpin loop. Two nested stems pay that once, not twice. So we measure the
interaction directly:

    J[s,t] = E({s,t}) - h[s] - h[t]

This is a two-body cluster expansion. It is **exact for any pair** of stems.
Without it the model stops after one stem and lands about 4 kcal/mol short.

In [8]:
# h and J already came from build_encoding above.
print('stem energies h (kcal/mol):')
print(np.round(h, 2))
print(f'\nclashing pairs: {(J > 0).sum()} out of {n*(n-1)//2}')
print('penalty size:', J.max())
print('most negative interaction:', round(float(J.min()), 2),
      '<- nested stems helping each other')

stem energies h (kcal/mol):
[-3.7   0.2   0.5   0.5   1.6   2.    0.2   1.7   0.9   5.46  1.9   4.13
  5.    4.1   2.8   4.3   1.4   3.9   4.4   2.9   2.1   2.3 ]

clashing pairs: 184 out of 231
penalty size: 16.380000114440918
most negative interaction: -3.7 <- nested stems helping each other


### Diagnostic: is the two-body expansion working?

The first check must pass exactly. The second shows what the naive additive
version (J = 0 for compatible stems) would have given you.

In [9]:
# Check 1: for every compatible PAIR, the QUBO must match ViennaRNA exactly.
worst = 0.0
for a in range(n):
    for b in range(a+1, n):
        if J[a, b] > 0:      # clashing pair, skip
            continue
        x = np.zeros(n); x[a] = x[b] = 1
        q = float(rna.qubo_energy(h, J, x)[0])
        v = rna.eval_structure(seq, rna.stems_to_dotbracket(len(seq), [stems[a], stems[b]]))
        worst = max(worst, abs(q - v))
print('max |QUBO - ViennaRNA| over compatible pairs:', round(worst, 8))
print('-> must be 0. If not, the encoding is broken.')

max |QUBO - ViennaRNA| over compatible pairs: 0.0
-> must be 0. If not, the encoding is broken.


In [10]:
# Check 2: how much does the two-body term actually buy us?
h0, J0 = rna.build_qubo(seq, stems, two_body=False)   # naive additive version
E0 = energy.full_energy_vector(h0, J0, dtype=np.float64)
k0, _ = baselines.brute_force(E0)
db0 = rna.decode_bits(seq, stems, energy.index_to_bits(np.array([k0]), n)[0], h=h0)

print('naive (J=0)   :', db0)
print('              ', round(rna.eval_structure(seq, db0), 2), 'kcal/mol')
print('two-body      : (computed in the next cell)')
print('ViennaRNA MFE :', round(refs['challenge44'][1], 2), 'kcal/mol')

naive (J=0)   : .(((((((............................))))))).
               -3.7 kcal/mol
two-body      : (computed in the next cell)
ViennaRNA MFE : -7.9 kcal/mol


## The energy of every possible answer

Because the Hamiltonian is diagonal, we can just list the energy of all 2^n
bitstrings once and reuse it. `full_energy_vector` does this with numpy views,
so it never builds a big temporary array.

In [11]:
E = energy.full_energy_vector(h, J, dtype=np.float64)
print("states:", len(E), " memory:", E.nbytes/1e6, "MB")

# Check it against a direct calculation. These must agree exactly.
bits_all = energy.index_to_bits(np.arange(len(E)), n)
E_direct = rna.qubo_energy(h, J, bits_all)
print("max difference:", np.abs(E - E_direct).max(), "(should be 0)")

states: 4194304  memory: 33.554432 MB
max difference: 0.0 (should be 0)


## Does the QUBO minimum equal the true MFE?

This is the single most important check in the whole project. If the QUBO's
best bitstring does not decode to the ViennaRNA MFE, the encoding is wrong and
no amount of clever quantum work will fix it.

In [12]:
k_best, e_best = baselines.brute_force(E)
bits_best = energy.index_to_bits(np.array([k_best]), n)[0]
db_pred = rna.decode_bits(seq, stems, bits_best, h=h)

ref_db, ref_e = refs["challenge44"]
print("QUBO best bitstring :", bits_best)
print("decoded structure   :", db_pred)
print("ViennaRNA MFE       :", ref_db)
print()
print("our energy (Vienna) :", round(rna.eval_structure(seq, db_pred), 2))
print("reference MFE       :", round(ref_e, 2))
print("gap                 :", round(rna.eval_structure(seq, db_pred) - ref_e, 2), "kcal/mol")
print()
metrics.print_row(metrics.report(seq, db_pred, ref_db,
                                 rna.eval_structure(seq, db_pred), ref_e))

QUBO best bitstring : [1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
decoded structure   : .(((((((..((((..(((........)))))))..))))))).
ViennaRNA MFE       : .(((((((..((((...(((....)))...))))..))))))).

our energy (Vienna) : -7.0
reference MFE       : -7.9
gap                 : 0.9 kcal/mol

  length  44 | gap  +0.90 kcal/mol | F1 0.79 | exact False


### Reranking: let ViennaRNA pick the winner

We take the lowest-energy bitstrings the QUBO offers, decode each into a real
structure, and score them exactly. Repair maps many bitstrings onto the same
structure, so a few hundred candidates collapse to far fewer evaluations.

In [13]:
db_rr, e_rr, seen = rna.best_from_energy_vector(seq, stems, E, n,
                                                top_k=200, h=h, verbose=True)
print()
print('QUBO argmin only  :', round(rna.eval_structure(seq, db_pred), 2))
print('after reranking   :', round(e_rr, 2))
print('ViennaRNA MFE     :', round(ref_e, 2))
print('remaining gap     :', round(e_rr - ref_e, 2), 'kcal/mol')
print()
metrics.print_row(metrics.report(seq, db_rr, ref_db, e_rr, ref_e))

  200 candidates -> 126 distinct structures
  best: -7.00 kcal/mol

QUBO argmin only  : -7.0
after reranking   : -7.0
ViennaRNA MFE     : -7.9
remaining gap     : 0.9 kcal/mol

  length  44 | gap  +0.90 kcal/mol | F1 0.79 | exact False


A gap of a few tenths here is the **two-body limit**, not a search failure.
A QUBO is quadratic, so it can only hold pair interactions between stems.
With three or more stems the true energy has higher-order terms that no
QUBO can represent. That is a property of QUBOs, and it applies to real
hardware too.

The standard fix is below: let the quantum part narrow millions of
bitstrings down to a few hundred candidates, then have ViennaRNA score
those exactly. Cheap, honest, and it is how hybrid algorithms normally work.

## Repair: every bitstring becomes a real structure

A random bitstring may pick clashing stems. Instead of throwing it away, we
keep the best stems and drop the ones that clash. This means the model never
wastes effort on invalid answers.

In [14]:
rng = np.random.default_rng(0)
for _ in range(3):
    b = rng.integers(0, 2, size=n)
    print("bits    :", b)
    print("repaired:", rna.decode_bits(seq, stems, b, h=h))
    print()

bits    : [1 1 1 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 1]
repaired: .(((((((.((((.....))))..............))))))).

bits    : [1 0 0 1 1 0 1 1 1 0 0 1 0 1 0 0 0 0 0 0 0 0]
repaired: .(((((((..((((.........)))).........))))))).

bits    : [0 1 1 1 0 1 1 0 0 1 1 1 0 1 1 1 1 1 1 0 1 0]
repaired: ...((((...))))(((.......((((........)))).)))



In [15]:
cu.save("encoding", {"seq": seq, "stems": stems, "h": h, "J": J,
                     "min_len": MIN_LEN, "n": n})
print("n =", n)

saved results/encoding.pkl
n = 22


## What we learned

- Stems instead of base pairs cut the qubit count by roughly 5x.
- `min_len` is the qubit-count knob.
- The QUBO minimum gives us the **encoding error floor** to report.
- Next: build the quantum circuit and let a GPT design it.

In [16]:
# Everything saved so far. These files live in your Drive, so the next
# notebook will find them even after this runtime shuts down.
cu.show_results()

# Uncomment to download a copy to your computer:
# cu.download_results()

file                        size   from notebook
--------------------------------------------------
encoding.pkl                5.2K   02
refs.pkl                    0.5K   01


In [17]:
# Does the untrimmed 27-stem encoding do better?
stems27 = rna.find_stems(seq, min_len=3)
h27, J27 = rna.build_qubo(seq, stems27)
E27 = energy.full_energy_vector(h27, J27, dtype=np.float32)   # ~540 MB, ~1 min
db27, e27, _ = rna.best_from_energy_vector(seq, stems27, E27, len(stems27),
                                           top_k=2000, h=h27)
print("22 qubits (trimmed):", -7.0)
print("27 qubits (full)   :", round(e27, 2))
print("ViennaRNA MFE      :", round(ref_e, 2))

22 qubits (trimmed): -7.0
27 qubits (full)   : -7.9
ViennaRNA MFE      : -7.9


So 27 qubits give the exact answer but we will use 22 qubits just due to computational resources.